In [1]:
# =============================================================================
# mumax3 Simulation Output Preprocessing
# =============================================================================
# This notebook preprocesses micromagnetic simulation results from mumax3.
# It reads time-domain magnetization data (.ovf files), applies a Fast Fourier
# Transform (FFT) along the time axis for each spatial point, and saves the
# resulting frequency-domain data as compressed .npz files for further analysis
# (e.g., spin-wave dispersion, inelastic scattering spectra).
# =============================================================================

# --- Standard scientific libraries ---
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline                         # Render plots inline in the notebook
import multiprocessing as mp               # Available for parallel processing (not directly used here)
import matplotlib.colors as colors         # Extended colormap utilities for plots
from os import path                        # OS-level path utilities (e.g., file existence checks)
import sys

# --- mumax3 post-processing library (mumax3PP) ---
import mumax3PP.ovf as ovf                             # Reader for mumax3 .ovf magnetization output files
import mumax3PP.parameters as parameters               # Simulation parameter configuration
import mumax3PP.fft_across_xyzm as FFT_across_xyzm    # Multi-threaded FFT over spatial+component dimensions

from IPython.display import clear_output   # Utility to clear cell output during loops

import glob                                # Unix-style pathname pattern expansion


def getDT(times):
    """
    Estimate the time step (dt) from an array of simulation output timestamps
    using a linear least-squares fit.

    This is more robust than simply taking the difference between two adjacent
    time points, as it accounts for minor floating-point irregularities in the
    saved timestamps.

    Parameters
    ----------
    times : array-like
        1D array of simulation time stamps (in seconds).

    Returns
    -------
    float
        The fitted time step dt (slope of the linear fit), in seconds.
    """
    xdata = range(len(times))              # Integer indices as the x-axis for fitting

    from scipy.optimize import curve_fit

    def func(x, a, b):
        """Linear model: a*x + b, where a = dt and b = t0."""
        return a * x + b

    # Initial guess for dt: total time span divided by number of intervals
    dt = np.abs(times[-1] - times[1]) / (len(times) - 1)

    # Fit the linear model to extract the best-estimate dt
    popt, pcov = curve_fit(func, xdata, times, p0=[dt, 0])

    # Diagnostic plots (commented out, useful for debugging timestamp irregularities)
    #     plt.plot(xdata, times)
    #     plt.plot(xdata, func(xdata, *popt))

    return popt[0]   # Return the fitted slope = dt

In [ ]:
# =============================================================================
# Cell 2: Discover simulation output directories
# =============================================================================
# Use glob to find all mumax3 output folders matching the naming pattern:
#   B_300mT_edge_f_<freq>GHz_amp_0.0300.out
# These correspond to simulations at B = 300 mT with a sinusoidal edge excitation
# at varying frequencies (GHz range) and fixed amplitude 0.03.
# The sorted() call ensures processing in ascending frequency order.

dirLi = glob.glob(r"D:\mumax3\inelastic_scattering\k_neg\edge_modes\B_300mT_edge_f_*GHz_amp_0.0300.out")
dirLi = sorted(dirLi)   # Sort alphabetically → ascending excitation frequency

# Print the discovered directories for verification
for i in dirLi:
    print(i)

In [ ]:
# =============================================================================
# Cell 3: Main preprocessing loop — load, FFT, save, and preview
# =============================================================================
# For each simulation directory:
#   1. Check if the FFT output file already exists (skip if so).
#   2. Load the time-domain magnetization data from .ovf files.
#   3. Apply a multithreaded FFT along the time axis at every spatial point.
#   4. Save the frequency-domain result as a compressed .npz file.
#   5. Plot the integrated spectral amplitude vs. frequency for a quick preview.
# =============================================================================

i = 1     # Counter for progress reporting

# Magnetization component to process: 'z' corresponds to the out-of-plane component Mz
comp = "z"

for path0 in dirLi:

    # Progress indicator: current / total
    print("{}/{}".format(i, np.shape(dirLi)[0]))

    # Construct the expected output filename for the FFT result.
    # Naming convention: <original_dir>_M<comp>_fzyxm.npz
    # 'fzyxm' indicates the axis order: frequency, z, y, x, magnetization component
    npzFile = path0[:-4] + "_M{}_fzyxm.npz".format(comp)

    print(npzFile)
    print(path.isfile(npzFile))   # True if this file was already computed

    # --- Skip condition (currently disabled) ---
    # Change `False` to `path.isfile(npzFile)` to enable skipping already-processed files.
    # This is useful for resuming interrupted batch runs.
    if False:  # path.isfile(npzFile):
        print(f"file {npzFile} already exists.")

    else:
        # Load simulation parameters: read magnetization vector field (head='m')
        # Additional options (commented out): spatial range restriction or time cutoff
        parms = parameters.ovfParms(head='m')   # head='m' → full magnetization vector
                                                # Alternative: head='m_z_xrange200' for spatial subset
                                                # tStop=-50 to truncate time series

        # Load all .ovf snapshots from the simulation directory.
        # M_txyzm.array has shape (T, nz, ny, nx, 3): time × spatial grid × vector components
        # M_txyzm.time contains the corresponding simulation timestamps in seconds.
        M_txyzm = ovf.OvfFile(path0, parms)
        print("==> data loaded, fft calculation...")

        # Apply FFT along the time axis using multiple threads for efficiency.
        # Only the first 800 time steps are used ([:800]) to ensure a uniform
        # length across all simulations and avoid transient startup artifacts.
        # The result replaces the time-domain array with frequency-domain amplitudes,
        # and M_txyzm.time is updated to contain the corresponding frequency bins.
        M_txyzm.array, M_txyzm.time = FFT_across_xyzm.FFT_across_xyzm_threads(
            M_txyzm.array[:800, :, :, :, :],   # Truncate to first 800 time steps
            (M_txyzm.time)[:]                   # Full time array for frequency axis calculation
        )
        # Alternative: single-threaded FFT (slower, kept for reference)
        #     M_txyzm.array, M_txyzm.time = FFT_across_xyzm.FFT_across_xyzm(M_txyzm.array, (M_txyzm.time))

        print("==> fft calculated, saving...")

        # Save the frequency-domain data to a compressed .npz archive.
        # Output file includes the component label and axis ordering in its name.
        M_txyzm.save(path0[:-4] + "_M{}_fzyxm.npz".format(comp))

        # Optional: free memory after saving (commented out; object goes out of scope at loop end)
        #         del(M_txyzm)

        print("==> done.")

    # Blank lines for visual separation in notebook output
    print("   ")
    print("   ")
    print("   ")

    # --- Quick diagnostic plot ---
    # Plot the spectral power (absolute FFT amplitude) integrated over all spatial
    # positions at the sample edge (last y-index, -1) for the y-component of M (index 1).
    # Frequency axis is converted from Hz to GHz (*1e-9).
    # The vertical line at 13 GHz marks the expected spin-wave resonance for reference.
    plt.plot(
        (M_txyzm.time)[1:] * 1e-9,                            # Frequency axis in GHz (skip DC bin)
        np.abs(np.sum(M_txyzm.array[1:, 0, :, -1, 1], axis=(1)))  # |FFT| summed over x, at edge (y=-1), My component
    )
    plt.xlabel(r"frequency (GHz)")
    plt.axvline(x=13)   # Reference line at 13 GHz (expected resonance / edge mode frequency)
    plt.show()

    i += 1   # Increment progress counter


print("DONE")